# Bangladeshi Taka Note Detection Using YOLOv8

**Project Overview:** Build an object detection model for Bangladeshi Taka currency notes and coins.

**Tools:** Roboflow, Ultralytics YOLOv8, Google Colab

**Dataset:** [Bangladeshi Currency Detection - Roboflow](https://universe.roboflow.com/tanvirtain/bangladeshi-currency-detection)

**Author:** Emon Rahman

---

## Section 1: Environment Setup & Installation

In [ ]:
# first let's install the libraries we need
!pip install ultralytics roboflow pyyaml -q

In [ ]:
# importing libraries
import os
import glob
import random
import shutil
import yaml
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches
from PIL import Image

from ultralytics import YOLO
from roboflow import Roboflow

# checking gpu
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found, will use CPU")

In [ ]:
# mounting drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB = True
    print("Running on Google Colab!")
except:
    COLAB = False
    print("Not running on Colab")

---
## Section 2: Dataset Collection (20 Marks)

For this project, I collected images of Bangladeshi Taka notes:
- **2, 5, 10, 20, 50, 100, 200, 500, and 1000 Taka**

**How I collected the images:**
- Took photos with my phone in different lighting and backgrounds
- Gathered images from Google and online sources
- Different angles, surfaces, and orientations

**Annotation was done using Roboflow** and exported in YOLO format.

In [ ]:
# downloading dataset from roboflow
# I annotated images on Roboflow and exported in YOLOv8 format

rf = Roboflow(api_key="e4Johe0jbj4e1XJ7o4qK")
project = rf.workspace("tanvirtain").project("bangladeshi-currency-detection")
version = project.version(1)
dataset = version.download("yolov8")

DATASET_DIR = dataset.location
print(f"\nDataset downloaded to: {DATASET_DIR}")

In [ ]:
print("Folders in dataset:")
for item in sorted(os.listdir(DATASET_DIR)):
    print(f"  {item}")

val_dir = os.path.join(DATASET_DIR, 'val')
valid_dir = os.path.join(DATASET_DIR, 'valid')
if os.path.exists(val_dir) and not os.path.exists(valid_dir):
    os.rename(val_dir, valid_dir)

valid_img_check = os.path.join(DATASET_DIR, 'valid', 'images')
test_img_check = os.path.join(DATASET_DIR, 'test', 'images')
has_valid = os.path.exists(valid_img_check) and len(glob.glob(os.path.join(valid_img_check, '*.*'))) > 0
has_test = os.path.exists(test_img_check) and len(glob.glob(os.path.join(test_img_check, '*.*'))) > 0

if not has_valid:
    print("Splitting train into train/valid/test...")
    train_img_dir = os.path.join(DATASET_DIR, 'train', 'images')
    train_lbl_dir = os.path.join(DATASET_DIR, 'train', 'labels')
    all_images = glob.glob(os.path.join(train_img_dir, '*.*'))
    random.seed(42)
    random.shuffle(all_images)
    n = len(all_images)
    train_end = int(n * 0.7)
    val_end = train_end + int(n * 0.2)
    valid_imgs = all_images[train_end:val_end]
    test_imgs = all_images[val_end:]
    for split_name, split_imgs in [('valid', valid_imgs), ('test', test_imgs)]:
        img_out = os.path.join(DATASET_DIR, split_name, 'images')
        lbl_out = os.path.join(DATASET_DIR, split_name, 'labels')
        os.makedirs(img_out, exist_ok=True)
        os.makedirs(lbl_out, exist_ok=True)
        for img_path in split_imgs:
            shutil.move(img_path, img_out)
            lbl_name = os.path.splitext(os.path.basename(img_path))[0] + '.txt'
            lbl_path = os.path.join(train_lbl_dir, lbl_name)
            if os.path.exists(lbl_path):
                shutil.move(lbl_path, lbl_out)
        print(f"  {split_name}: {len(split_imgs)} images")
    print(f"  train: {train_end} images")
elif not has_test:
    print("Creating test split from validation...")
    test_img_dir = os.path.join(DATASET_DIR, 'test', 'images')
    test_lbl_dir = os.path.join(DATASET_DIR, 'test', 'labels')
    os.makedirs(test_img_dir, exist_ok=True)
    os.makedirs(test_lbl_dir, exist_ok=True)
    valid_imgs = glob.glob(os.path.join(DATASET_DIR, 'valid', 'images', '*.*'))
    random.seed(42)
    test_samples = random.sample(valid_imgs, max(1, len(valid_imgs) // 3))
    for img_path in test_samples:
        shutil.copy2(img_path, test_img_dir)
        lbl_name = os.path.splitext(os.path.basename(img_path))[0] + '.txt'
        lbl_path = os.path.join(DATASET_DIR, 'valid', 'labels', lbl_name)
        if os.path.exists(lbl_path):
            shutil.copy2(lbl_path, test_lbl_dir)
    print(f"  test: {len(test_samples)} images")
else:
    print("All splits exist.")

yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)
data_config['path'] = os.path.abspath(DATASET_DIR)
data_config['train'] = 'train/images'
data_config['val'] = 'valid/images'
data_config['test'] = 'test/images'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)
print("data.yaml updated")
with open(yaml_path, 'r') as f:
    print(f.read())

In [ ]:
# checking folder structure

print("Dataset Folder Structure:")
print("=" * 50)
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    folder_name = os.path.basename(root)
    if files:
        print(f"{indent}{folder_name}/ ({len(files)} files)")
    else:
        print(f"{indent}{folder_name}/")

In [ ]:
# getting class names

names_raw = data_config['names']
if isinstance(names_raw, dict):
    CLASS_NAMES = [names_raw[i] for i in sorted(names_raw.keys())]
else:
    CLASS_NAMES = list(names_raw)

NUM_CLASSES = len(CLASS_NAMES)
print(f"Number of classes: {NUM_CLASSES}")
for i, name in enumerate(CLASS_NAMES):
    print(f"  Class {i}: {name}")

### Dataset Statistics

Let's count images per split and per class.

In [ ]:
# counting images per split

splits = ['train', 'valid', 'test']
split_counts = {}
for split in splits:
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    if os.path.exists(img_dir):
        images = glob.glob(os.path.join(img_dir, '*.jpg')) + \
                 glob.glob(os.path.join(img_dir, '*.png')) + \
                 glob.glob(os.path.join(img_dir, '*.jpeg'))
        split_counts[split] = len(images)
    else:
        split_counts[split] = 0

print("Dataset Split Statistics:")
print("=" * 40)
total = sum(split_counts.values())
for split, count in split_counts.items():
    pct = (count / total * 100) if total > 0 else 0
    print(f"{split:10s}: {count:5d} images ({pct:.1f}%)")
print(f"{'Total':10s}: {total:5d} images")

In [ ]:
# counting annotations per class

def count_classes_in_labels(label_dir, class_names):
    class_counts = Counter()
    if not os.path.exists(label_dir):
        return class_counts
    for lf in glob.glob(os.path.join(label_dir, '*.txt')):
        with open(lf, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cid = int(parts[0])
                    if cid < len(class_names):
                        class_counts[class_names[cid]] += 1
    return class_counts

print("Annotations per class:")
print("=" * 50)
total_counts = Counter()
for split in splits:
    label_dir = os.path.join(DATASET_DIR, split, 'labels')
    counts = count_classes_in_labels(label_dir, CLASS_NAMES)
    total_counts += counts
    print(f"\n{split.upper()} set:")
    for cn in CLASS_NAMES:
        print(f"  {cn:20s}: {counts.get(cn, 0):4d}")
print(f"\nTOTAL:")
for cn in CLASS_NAMES:
    print(f"  {cn:20s}: {total_counts.get(cn, 0):4d}")

In [ ]:
# plotting class distribution

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
classes = list(total_counts.keys()) if total_counts else CLASS_NAMES
counts_list = [total_counts.get(c, 0) for c in classes]
colors = plt.cm.Set3(np.linspace(0, 1, len(classes)))
axes[0].bar(range(len(classes)), counts_list, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xticks(range(len(classes)))
axes[0].set_xticklabels(classes, rotation=45, ha='right')
axes[0].set_xlabel('Taka Denomination')
axes[0].set_ylabel('Number of Annotations')
axes[0].set_title('Class Distribution')
sl = [s for s in split_counts if split_counts[s] > 0]
sv = [split_counts[s] for s in sl]
axes[1].pie(sv, labels=sl, autopct='%1.1f%%', colors=['#66b3ff','#99ff99','#ff9999'][:len(sl)], startangle=90)
axes[1].set_title('Train / Valid / Test Split')
plt.tight_layout()
plt.savefig('dataset_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# showing some training images with boxes

img_dir = os.path.join(DATASET_DIR, 'train', 'images')
label_dir = os.path.join(DATASET_DIR, 'train', 'labels')
image_files = glob.glob(os.path.join(img_dir, '*.jpg')) + glob.glob(os.path.join(img_dir, '*.png')) + glob.glob(os.path.join(img_dir, '*.jpeg'))
num_samples = min(9, len(image_files))
samples = random.sample(image_files, num_samples)
cols = 3
rows = max(1, (num_samples + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = np.array(axes).reshape(rows, cols)
cmap = plt.cm.tab10
for idx, img_path in enumerate(samples):
    row, col = idx // cols, idx % cols
    ax = axes[row][col]
    img = np.array(Image.open(img_path).convert('RGB'))
    h, w = img.shape[:2]
    ax.imshow(img)
    lp = os.path.join(label_dir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')
    if os.path.exists(lp):
        with open(lp, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cid = int(parts[0])
                    xc, yc = float(parts[1])*w, float(parts[2])*h
                    bw, bh = float(parts[3])*w, float(parts[4])*h
                    x1, y1 = xc - bw/2, yc - bh/2
                    color = cmap(cid / max(NUM_CLASSES, 1))
                    ax.add_patch(patches.Rectangle((x1,y1), bw, bh, linewidth=2, edgecolor=color, facecolor='none'))
                    lt = CLASS_NAMES[cid] if cid < NUM_CLASSES else str(cid)
                    ax.text(x1, max(y1-5,10), lt, color='white', fontsize=8, bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.7))
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')
for idx in range(num_samples, rows*cols):
    axes[idx//cols][idx%cols].axis('off')
plt.suptitle('Sample Training Images with Bounding Boxes', fontsize=14)
plt.tight_layout()
plt.savefig('sample_images_train.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3: Data Annotation & Dataset Preparation (20 Marks)

### Annotation Process
1. Uploaded images to Roboflow
2. Drew bounding boxes around each note
3. Labeled with denomination
4. Applied augmentations
5. Exported in **YOLOv8 format**

### YOLO Format: `class_id x_center y_center width height` (normalized 0-1)

In [ ]:
# looking at a sample label file

label_dir = os.path.join(DATASET_DIR, 'train', 'labels')
label_files = glob.glob(os.path.join(label_dir, '*.txt'))
if label_files:
    print(f"Sample: {os.path.basename(label_files[0])}")
    print("=" * 50)
    with open(label_files[0], 'r') as f:
        print(f.read())
    print("Format: class_id  x_center  y_center  width  height")

In [ ]:
# split function (already split from roboflow)

def create_dataset_split(src_img, src_lbl, out_dir, train_r=0.7, val_r=0.2, test_r=0.1):
    imgs = glob.glob(os.path.join(src_img, '*.jpg')) + glob.glob(os.path.join(src_img, '*.png'))
    random.seed(42)
    random.shuffle(imgs)
    n = len(imgs)
    te, ve = int(n*train_r), int(n*train_r) + int(n*val_r)
    for name, subset in [('train',imgs[:te]),('valid',imgs[te:ve]),('test',imgs[ve:])]:
        for d in ['images','labels']:
            os.makedirs(os.path.join(out_dir,name,d), exist_ok=True)
        for ip in subset:
            shutil.copy2(ip, os.path.join(out_dir,name,'images'))
            lp = os.path.join(src_lbl, os.path.splitext(os.path.basename(ip))[0]+'.txt')
            if os.path.exists(lp): shutil.copy2(lp, os.path.join(out_dir,name,'labels'))
        print(f"{name}: {len(subset)} images")

print("Dataset already split from Roboflow.")

In [ ]:
# checking if all images have labels

print("Checking image-label pairs:")
print("=" * 50)
for split in ['train', 'valid', 'test']:
    idir = os.path.join(DATASET_DIR, split, 'images')
    ldir = os.path.join(DATASET_DIR, split, 'labels')
    if not os.path.exists(idir): continue
    imgs = glob.glob(os.path.join(idir,'*.jpg'))+glob.glob(os.path.join(idir,'*.png'))+glob.glob(os.path.join(idir,'*.jpeg'))
    miss = sum(1 for i in imgs if not os.path.exists(os.path.join(ldir, os.path.splitext(os.path.basename(i))[0]+'.txt')))
    print(f"{split:8s}: {len(imgs)} images, missing labels: {miss} [{'OK' if miss==0 else 'WARNING'}]")

---
## Section 4: Model Training (30 Marks)

### Configuration:
- **Model:** YOLOv8n (pretrained on COCO)
- **Epochs:** 50
- **Batch Size:** 16
- **Image Size:** 640x640

In [ ]:
# loading yolov8 model

model = YOLO('yolov8n.pt')
print("Model loaded: YOLOv8 Nano (~3.2M params)")

In [ ]:
# training on our dataset

EPOCHS = 50
BATCH_SIZE = 16
IMG_SIZE = 640
PATIENCE = 15

print(f"Training: {EPOCHS} epochs, batch {BATCH_SIZE}, {IMG_SIZE}x{IMG_SIZE}")
print("=" * 50)

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=PATIENCE,
    save=True,
    save_period=10,
    project='taka_detection',
    name='train_v1',
    exist_ok=True,
    pretrained=True,
    verbose=True
)
print("\nTraining completed!")

In [ ]:
# checking output files

train_dir = 'taka_detection/train_v1'
if os.path.exists(train_dir):
    for item in sorted(os.listdir(train_dir)):
        p = os.path.join(train_dir, item)
        if os.path.isfile(p):
            print(f"  {item} ({os.path.getsize(p)/1024/1024:.2f} MB)")
        else:
            print(f"  {item}/")

In [ ]:
# training curves

rp = os.path.join(train_dir, 'results.png')
if os.path.exists(rp):
    plt.figure(figsize=(20,10))
    plt.imshow(Image.open(rp))
    plt.axis('off')
    plt.title('Training Results')
    plt.show()

In [ ]:
# confusion matrix

cm = os.path.join(train_dir, 'confusion_matrix.png')
if os.path.exists(cm):
    plt.figure(figsize=(12,10))
    plt.imshow(Image.open(cm))
    plt.axis('off')
    plt.title('Confusion Matrix')
    plt.show()

In [ ]:
# training log

csv_path = os.path.join(train_dir, 'results.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    print(f"Trained {len(df)} epochs")
    map50_col = [c for c in df.columns if 'mAP50' in c and '50-95' not in c]
    map5095_col = [c for c in df.columns if 'mAP50-95' in c]
    prec_col = [c for c in df.columns if 'precision' in c]
    rec_col = [c for c in df.columns if 'recall' in c]
    if map50_col:
        print(f"Best mAP50: {df[map50_col[0]].max():.4f} (epoch {df[map50_col[0]].idxmax()+1})")
    if map5095_col: print(f"Best mAP50-95: {df[map5095_col[0]].max():.4f}")
    if prec_col: print(f"Best Precision: {df[prec_col[0]].max():.4f}")
    if rec_col: print(f"Best Recall: {df[rec_col[0]].max():.4f}")
    print("\nLast 5 epochs:")
    print(df.tail())

In [ ]:
# plotting metrics

if os.path.exists(csv_path):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    tb = [c for c in df.columns if 'train' in c and 'box' in c]
    vb = [c for c in df.columns if 'val' in c and 'box' in c]
    tc = [c for c in df.columns if 'train' in c and 'cls' in c]
    vc = [c for c in df.columns if 'val' in c and 'cls' in c]
    if tb and vb:
        axes[0][0].plot(df[tb[0]], label='Train'); axes[0][0].plot(df[vb[0]], label='Val')
        axes[0][0].set_title('Box Loss'); axes[0][0].legend(); axes[0][0].grid(True, alpha=0.3)
    if tc and vc:
        axes[0][1].plot(df[tc[0]], label='Train'); axes[0][1].plot(df[vc[0]], label='Val')
        axes[0][1].set_title('Cls Loss'); axes[0][1].legend(); axes[0][1].grid(True, alpha=0.3)
    if map50_col and map5095_col:
        axes[1][0].plot(df[map50_col[0]], label='mAP50'); axes[1][0].plot(df[map5095_col[0]], label='mAP50-95')
        axes[1][0].set_title('mAP'); axes[1][0].legend(); axes[1][0].grid(True, alpha=0.3)
    if prec_col and rec_col:
        axes[1][1].plot(df[prec_col[0]], label='Precision'); axes[1][1].plot(df[rec_col[0]], label='Recall')
        axes[1][1].set_title('Precision & Recall'); axes[1][1].legend(); axes[1][1].grid(True, alpha=0.3)
    plt.suptitle('Training Metrics')
    plt.tight_layout()
    plt.savefig('training_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# model weights

best_weights = os.path.join(train_dir, 'weights', 'best.pt')
print(f"Best weights: {best_weights}")
if os.path.exists(best_weights):
    print(f"Size: {os.path.getsize(best_weights)/1024/1024:.2f} MB")

---
## Section 5: Model Evaluation (20 Marks)

Evaluating on the **test dataset**.

In [ ]:
# loading best model
best_model = YOLO(best_weights)
print("Best model loaded")

In [ ]:
# evaluating on test set

eval_results = best_model.val(
    data=yaml_path, split='test', imgsz=IMG_SIZE, batch=BATCH_SIZE,
    project='taka_detection', name='eval_test', exist_ok=True, verbose=True
)
print("Evaluation completed!")

In [ ]:
# printing results

print("=" * 60)
print("       TEST SET RESULTS")
print("=" * 60)
rd = eval_results.results_dict
print(f"Precision:   {rd.get('metrics/precision(B)',0):.4f}")
print(f"Recall:      {rd.get('metrics/recall(B)',0):.4f}")
print(f"mAP@50:      {rd.get('metrics/mAP50(B)',0):.4f}")
print(f"mAP@50-95:   {rd.get('metrics/mAP50-95(B)',0):.4f}")
print("\nPer-Class AP@50:")
try:
    for i, ci in enumerate(eval_results.ap_class_index):
        cn = CLASS_NAMES[int(ci)] if int(ci) < NUM_CLASSES else f"class_{ci}"
        print(f"  {cn:20s}: {eval_results.box.ap50[i]:.4f}")
except: pass

In [ ]:
# evaluation plots

eval_dir = 'taka_detection/eval_test'
for fn, t in [('confusion_matrix.png','Confusion Matrix'),('F1_curve.png','F1 Curve'),('PR_curve.png','PR Curve')]:
    fp = os.path.join(eval_dir, fn)
    if os.path.exists(fp):
        plt.figure(figsize=(10,8))
        plt.imshow(Image.open(fp))
        plt.axis('off')
        plt.title(t)
        plt.show()

---
## Section 6: Inference / Detection Results (10 Marks)

In [ ]:
# running predictions on test images

test_images_dir = os.path.join(DATASET_DIR, 'test', 'images')
best_model.predict(source=test_images_dir, imgsz=IMG_SIZE, conf=0.25, save=True,
                   project='taka_detection', name='predictions', exist_ok=True)
print("Predictions saved!")

In [ ]:
# showing prediction results

pred_dir = 'taka_detection/predictions'
pred_imgs = glob.glob(os.path.join(pred_dir,'*.jpg'))+glob.glob(os.path.join(pred_dir,'*.png'))+glob.glob(os.path.join(pred_dir,'*.jpeg'))
if pred_imgs:
    n = min(12, len(pred_imgs))
    samp = random.sample(pred_imgs, n)
    cols = 3; rows = max(1,(n+2)//3)
    fig, axes = plt.subplots(rows, cols, figsize=(18,6*rows))
    axes = np.array(axes).reshape(rows,cols)
    for i,p in enumerate(samp):
        axes[i//cols][i%cols].imshow(Image.open(p))
        axes[i//cols][i%cols].set_title(os.path.basename(p), fontsize=8)
        axes[i//cols][i%cols].axis('off')
    for i in range(n, rows*cols):
        axes[i//cols][i%cols].axis('off')
    plt.suptitle('Detection Results on Test Images', fontsize=16)
    plt.tight_layout()
    plt.savefig('detection_results.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# detailed results for some images

test_images = glob.glob(os.path.join(test_images_dir,'*.jpg'))+glob.glob(os.path.join(test_images_dir,'*.png'))+glob.glob(os.path.join(test_images_dir,'*.jpeg'))
print("Detailed Detection Results:")
print("=" * 60)
for ip in test_images[:5]:
    r = best_model.predict(ip, imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
    print(f"\nImage: {os.path.basename(ip)}")
    if len(r.boxes) > 0:
        for b in r.boxes:
            cid = int(b.cls[0])
            print(f"  {r.names.get(cid,'?')} | Conf: {float(b.conf[0]):.2%} | Box: {[int(x) for x in b.xyxy[0].tolist()]}")
    else:
        print("  No detections")

In [ ]:
# original vs detection comparison

if test_images:
    si = test_images[0]
    r = best_model.predict(si, imgsz=640, conf=0.25, verbose=False)[0]
    fig, (a1,a2) = plt.subplots(1,2,figsize=(16,6))
    a1.imshow(np.array(Image.open(si).convert('RGB')))
    a1.set_title('Original'); a1.axis('off')
    a2.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    a2.set_title('Detection'); a2.axis('off')
    plt.suptitle('Single Image Inference Demo')
    plt.tight_layout()
    plt.savefig('single_inference_demo.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Bonus: Coin Detection Extension (15 Marks)

Extending the model to detect **2 Taka** and **5 Taka** coins.

In [ ]:
# adding coin classes

CLASS_NAMES_WITH_COINS = CLASS_NAMES.copy() + ['2_taka_coin', '5_taka_coin']
print("Classes with coins:")
for i, n in enumerate(CLASS_NAMES_WITH_COINS):
    m = ' <-- NEW' if 'coin' in n else ''
    print(f"  {i}: {n}{m}")

In [ ]:
# new yaml with coin classes

coin_yaml = {'path': os.path.abspath(DATASET_DIR), 'train': 'train/images',
             'val': 'valid/images', 'test': 'test/images',
             'nc': len(CLASS_NAMES_WITH_COINS), 'names': CLASS_NAMES_WITH_COINS}
coin_yaml_path = os.path.join(DATASET_DIR, 'data_with_coins.yaml')
with open(coin_yaml_path, 'w') as f:
    yaml.dump(coin_yaml, f, default_flow_style=False)
print(f"Created data_with_coins.yaml ({len(CLASS_NAMES_WITH_COINS)} classes)")

In [ ]:
# training with coins

coin_model = YOLO('yolov8n.pt')
coin_model.train(data=coin_yaml_path, epochs=50, batch=16, imgsz=640,
                 patience=15, save=True, project='taka_detection',
                 name='train_with_coins', exist_ok=True, pretrained=True, verbose=True)
print("\nCoin training completed!")

In [ ]:
# evaluating coin model

cwt = 'taka_detection/train_with_coins/weights/best.pt'
if os.path.exists(cwt):
    cm = YOLO(cwt)
    ce = cm.val(data=coin_yaml_path, split='test', imgsz=640, batch=16,
               project='taka_detection', name='eval_coins', exist_ok=True)
    crd = ce.results_dict
    print("Coin Model Results:")
    print(f"  Precision: {crd.get('metrics/precision(B)',0):.4f}")
    print(f"  Recall:    {crd.get('metrics/recall(B)',0):.4f}")
    print(f"  mAP@50:    {crd.get('metrics/mAP50(B)',0):.4f}")
    print(f"  mAP@50-95: {crd.get('metrics/mAP50-95(B)',0):.4f}")

In [ ]:
# coin detection results

if os.path.exists(cwt):
    cm.predict(source=test_images_dir, imgsz=640, conf=0.25, save=True,
              project='taka_detection', name='predictions_with_coins', exist_ok=True)
    cpd = 'taka_detection/predictions_with_coins'
    cpi = glob.glob(os.path.join(cpd,'*.jpg'))+glob.glob(os.path.join(cpd,'*.png'))+glob.glob(os.path.join(cpd,'*.jpeg'))
    if cpi:
        n = min(6, len(cpi))
        s = random.sample(cpi, n)
        cols=3; rows=max(1,(n+2)//3)
        fig, axes = plt.subplots(rows,cols,figsize=(18,6*rows))
        axes = np.array(axes).reshape(rows,cols)
        for i,p in enumerate(s):
            axes[i//cols][i%cols].imshow(Image.open(p))
            axes[i//cols][i%cols].set_title(os.path.basename(p),fontsize=8)
            axes[i//cols][i%cols].axis('off')
        for i in range(n,rows*cols):
            axes[i//cols][i%cols].axis('off')
        plt.suptitle('Coin Detection Results (Bonus)', fontsize=16)
        plt.tight_layout()
        plt.savefig('coin_detection_results.png', dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# coin training curves

cri = 'taka_detection/train_with_coins/results.png'
if os.path.exists(cri):
    plt.figure(figsize=(20,10))
    plt.imshow(Image.open(cri))
    plt.axis('off')
    plt.title('Coin Model Training Curves')
    plt.show()

---
## Section 7: Summary & Submission

In [ ]:
# summary

print("=" * 70)
print("PROJECT SUMMARY")
print("=" * 70)
print(f"Dataset: {total} images, {NUM_CLASSES} classes")
print(f"Split: Train {split_counts.get('train',0)} | Valid {split_counts.get('valid',0)} | Test {split_counts.get('test',0)}")
print(f"Model: YOLOv8n, {EPOCHS} epochs, batch {BATCH_SIZE}, {IMG_SIZE}px")
print(f"Bonus: {len(CLASS_NAMES_WITH_COINS)} classes (with coins)")
print(f"\nDataset: https://universe.roboflow.com/tanvirtain/bangladeshi-currency-detection")

In [ ]:
# saving to drive

if COLAB:
    sd = '/content/drive/MyDrive/taka_detection_results'
    os.makedirs(sd, exist_ok=True)
    if os.path.exists(best_weights):
        shutil.copy2(best_weights, os.path.join(sd, 'best.pt'))
        print('Best weights saved to Drive')
    cwt = 'taka_detection/train_with_coins/weights/best.pt'
    if os.path.exists(cwt):
        shutil.copy2(cwt, os.path.join(sd, 'best_with_coins.pt'))
        print('Coin weights saved to Drive')
else:
    print('Weights in taka_detection/ folder')
print('\nProject complete!')